# Breast Cancer Biomarker Discovery via LASSO Regression

**Dataset:** GSE42568 — 104 breast cancer + 17 normal breast tissue transcriptomic profiles (Affymetrix U133 Plus 2.0).

**Objective:** Parse the GEO SOFT series matrix, verify sample alignment, standardise the expression matrix, and apply L1-regularised regression (LassoCV) to identify the most predictive gene probes for distinguishing cancer from normal tissue.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, Lasso

DATA_PATH = "data/GSE42568_series_matrix.txt"

## 1 — Data Parsing & Alignment Verification

We scan the metadata header to extract:
- **Target labels ($y$):** from the `!Sample_characteristics_ch1` row containing `"tissue:"`
- **Sample IDs:** from the `!Sample_geo_accession` row
- **Matrix start line:** the `!series_matrix_table_begin` marker

In [ ]:
# --- Parse metadata in a single pass ---
y = None
sample_ids = None
matrix_start_line = None

with open(DATA_PATH, "r") as f:
    for line_num, line in enumerate(f, start=1):
        # Target labels: only the "tissue:" row
        if line.startswith("!Sample_characteristics_ch1") and "tissue:" in line:
            fields = line.strip().split("\t")[1:]          # drop the row label
            y = np.array(
                [0 if "normal breast" in v else 1 for v in fields],
                dtype=int,
            )

        # Sample GEO accession IDs
        if line.startswith("!Sample_geo_accession"):
            sample_ids = [
                v.strip().strip('"') for v in line.strip().split("\t")[1:]
            ]

        # Matrix begin marker
        if line.startswith("!series_matrix_table_begin"):
            matrix_start_line = line_num
            break  # everything we need from the header is done

# --- Alignment verification ---
assert y is not None, "Could not find 'tissue:' row in metadata!"
assert sample_ids is not None, "Could not find Sample_geo_accession row!"
assert len(y) == len(sample_ids), (
    f"Length mismatch: {len(y)} labels vs {len(sample_ids)} sample IDs"
)

n_normal = int((y == 0).sum())
n_cancer = int((y == 1).sum())

print(f"Verification passed: {len(y)} labels and {len(sample_ids)} sample IDs")
print(f"  Normal breast: {n_normal}")
print(f"  Breast cancer: {n_cancer}")
print(f"Matrix data starts after line {matrix_start_line}")

## 2 — Load & Preprocess the Expression Matrix

1. Read the tab-delimited table below `!series_matrix_table_begin`.
2. Drop the trailing `!series_matrix_table_end` sentinel.
3. Set `ID_REF` as the index, then **transpose** so rows = samples, columns = probes.
4. Standardise with `StandardScaler`.

In [ ]:
# --- Load expression matrix ---
# skiprows: skip all header lines up to and including the table_begin marker
expr = pd.read_csv(
    DATA_PATH,
    sep="\t",
    skiprows=matrix_start_line,   # 0-indexed skip count = line number of marker
    index_col=0,                  # ID_REF as index
)

# Drop the !series_matrix_table_end sentinel (last row)
if expr.index[-1].startswith("!"):
    expr = expr.iloc[:-1]

# Transpose: rows → samples, columns → gene probes
X_df = expr.T

# Ensure all values are numeric (some edge-case nulls become NaN → fill with 0)
X_df = X_df.apply(pd.to_numeric, errors="coerce").fillna(0)

print(f"Transposed matrix shape: {X_df.shape}  (samples × probes)")

In [ ]:
# --- Standardise features ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_df)

print(f"Standardised matrix shape: {X_scaled.shape}")
print(f"Mean ≈ 0 check → first 5 col means: {X_scaled.mean(axis=0)[:5].round(6)}")
print(f"Std  ≈ 1 check → first 5 col stds:  {X_scaled.std(axis=0)[:5].round(6)}")

## 2b — PCA Diagnostic & Visualisation

Before modelling, we project the standardised data onto its first 2–3 principal components to:
- Confirm the cancer vs. normal separation exists in an **unsupervised** embedding (no label leakage).
- Check for batch effects, outliers, or unexpected sub-clusters that could bias downstream results.
- Quantify how much variance the top components capture.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# --- Fit PCA (keep enough components for the scree plot) ---
pca_full = PCA(n_components=min(20, X_scaled.shape[0]))
X_pca = pca_full.fit_transform(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

print(f"PC1 explains {explained[0]*100:.1f}% of variance")
print(f"PC2 explains {explained[1]*100:.1f}% of variance")
print(f"PC1+PC2 cumulative: {cumulative[1]*100:.1f}%")
print(f"Components needed for 80% variance: {np.searchsorted(cumulative, 0.80) + 1}")

In [ ]:
# --- Scree plot ---
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: individual explained variance
axes[0].bar(range(1, len(explained)+1), explained * 100,
            color='steelblue', edgecolor='white')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance Explained (%)')
axes[0].set_title('Scree Plot')

# Right: cumulative
axes[1].plot(range(1, len(cumulative)+1), cumulative * 100,
             'o-', color='steelblue', lw=2, markersize=5)
axes[1].axhline(80, ls='--', color='grey', lw=1, label='80% threshold')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()

fig.tight_layout()
plt.show()

In [ ]:
# --- 2-D scatter: PC1 vs PC2 coloured by class ---
labels_str = np.where(y == 0, 'Normal', 'Cancer')
colors = {'Normal': '#2ecc71', 'Cancer': '#e74c3c'}

fig, ax = plt.subplots(figsize=(8, 6))
for label in ['Normal', 'Cancer']:
    mask = labels_str == label
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=colors[label], label=label, s=60, alpha=0.75, edgecolors='white', lw=0.5)

ax.set_xlabel(f'PC1 ({explained[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({explained[1]*100:.1f}%)')
ax.set_title('PCA — Cancer vs Normal Tissue (Unsupervised)')
ax.legend(title='Tissue', frameon=True)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

print("\n→ If the two clusters separate cleanly, the biological signal is real")
print("  and not an artefact of batch effects or labelling errors.")

## 3 — LASSO Regression (LassoCV)

We use `LassoCV` with 5-fold cross-validation to automatically select the optimal regularisation strength $\alpha$.  
The L1 penalty drives most coefficients to exactly zero, performing **embedded feature selection**.

In [ ]:
# --- Fit LassoCV ---
lasso = LassoCV(cv=5, random_state=42, max_iter=10_000)
lasso.fit(X_scaled, y)

print(f"Optimal alpha (λ): {lasso.alpha_:.6f}")
print(f"Non-zero features: {(lasso.coef_ != 0).sum()}")

## 3b — Refined LASSO: Targeting 20–25 Biomarkers

The initial LassoCV selected ~91 features. PCA showed that **21 components** capture 80% of variance,
suggesting the data's intrinsic dimensionality is ~20. We now search for a stricter $\alpha$ that
yields exactly **20–25 non-zero genes**, using the CV-optimal $\alpha$ as a lower bound and binary search
to find the tightest penalty that stays in this target range.

In [ ]:
# --- Binary search for alpha that yields 20-25 non-zero coefficients ---
TARGET_MIN, TARGET_MAX = 20, 25

alpha_lo = lasso.alpha_    # CV-optimal (too many features)
alpha_hi = 0.5             # very strong penalty (near-zero features)

best_alpha = None
best_n_features = None

for _ in range(100):  # max iterations for binary search
    alpha_mid = (alpha_lo + alpha_hi) / 2
    model = Lasso(alpha=alpha_mid, max_iter=10_000)
    model.fit(X_scaled, y)
    n_feat = (model.coef_ != 0).sum()

    if TARGET_MIN <= n_feat <= TARGET_MAX:
        best_alpha = alpha_mid
        best_n_features = n_feat
        break
    elif n_feat > TARGET_MAX:
        alpha_lo = alpha_mid   # need more penalty
    else:
        alpha_hi = alpha_mid   # need less penalty

# Fallback: if binary search didn't land exactly in [20,25], use closest
if best_alpha is None:
    best_alpha = alpha_mid
    best_n_features = n_feat
    print(f"⚠ Could not land exactly in [{TARGET_MIN}, {TARGET_MAX}], closest: {n_feat} features")

# --- Fit final strict model ---
lasso_strict = Lasso(alpha=best_alpha, max_iter=10_000)
lasso_strict.fit(X_scaled, y)

print(f"CV-optimal alpha:     {lasso.alpha_:.6f}  →  {(lasso.coef_ != 0).sum()} genes")
print(f"Strict alpha:         {best_alpha:.6f}  →  {best_n_features} genes")
print(f"Alpha multiplier:     {best_alpha / lasso.alpha_:.1f}×")

In [ ]:
# --- Extract & display the refined biomarker panel ---
coef_strict = pd.Series(lasso_strict.coef_, index=X_df.columns, name="coefficient")
nonzero_strict = coef_strict[coef_strict != 0].copy()
nonzero_strict = nonzero_strict.reindex(nonzero_strict.abs().sort_values(ascending=False).index)

panel = nonzero_strict.to_frame()
panel.index.name = "Probe_ID"
panel["abs_coefficient"] = panel["coefficient"].abs()
panel["direction"] = np.where(panel["coefficient"] > 0, "↑ cancer", "↓ cancer")
panel["rank"] = range(1, len(panel) + 1)

# --- Quick accuracy check ---
from sklearn.metrics import accuracy_score
y_pred_strict = (lasso_strict.predict(X_scaled) > 0.5).astype(int)
acc_strict = accuracy_score(y, y_pred_strict)

print(f"Strict LASSO — {len(panel)} biomarker probes selected")
print(f"Training accuracy:  {acc_strict:.4f}")
print(f"R² (in-sample):     {lasso_strict.score(X_scaled, y):.4f}")
print()
print("=" * 65)
print(f"  REFINED BIOMARKER PANEL ({len(panel)} probes, α = {best_alpha:.6f})")
print("=" * 65)
panel

## 4 — Results: Top Biomarker Probes (Initial LassoCV)

Extract only the non-zero coefficients from the initial CV model, rank by absolute magnitude, and display the **top 20** most important probes.

In [ ]:
# --- Extract & rank non-zero coefficients ---
coef_series = pd.Series(lasso.coef_, index=X_df.columns, name="coefficient")
nonzero = coef_series[coef_series != 0].copy()
nonzero = nonzero.reindex(nonzero.abs().sort_values(ascending=False).index)

print(f"Total probes:           {len(coef_series):,}")
print(f"Non-zero (selected):    {len(nonzero):,}")
print(f"Zeroed-out (excluded):  {(coef_series == 0).sum():,}")
print()

# --- Display the top 20 biomarker probes ---
top20 = nonzero.head(20).to_frame()
top20.index.name = "Probe_ID"
top20["abs_coefficient"] = top20["coefficient"].abs()
top20["direction"] = np.where(top20["coefficient"] > 0, "↑ cancer", "↓ cancer")

print("=" * 60)
print("  TOP 20 LASSO-SELECTED BREAST CANCER BIOMARKER PROBES")
print("=" * 60)
top20

## 5 — Performance Verification

Sanity-check the model with R², classification accuracy, and the LassoCV alpha-path MSE curve.

In [ ]:
from sklearn.metrics import classification_report

# --- Scores ---
r2 = lasso.score(X_scaled, y)
y_pred = (lasso.predict(X_scaled) > 0.5).astype(int)
acc = accuracy_score(y, y_pred)

print(f"R² score (in-sample):      {r2:.4f}")
print(f"Training accuracy:         {acc:.4f}")
print(f"Optimal alpha (λ):         {lasso.alpha_:.6f}")
print(f"Non-zero / Total features: {(lasso.coef_ != 0).sum()} / {len(lasso.coef_)}")
print()
print(classification_report(y, y_pred, target_names=['Normal', 'Cancer']))

# --- Alpha-path MSE curve ---
mean_mse = np.mean(lasso.mse_path_, axis=1)
std_mse  = np.std(lasso.mse_path_, axis=1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogx(lasso.alphas_, mean_mse, color='steelblue', lw=2, label='Mean CV MSE')
ax.fill_between(lasso.alphas_, mean_mse - std_mse, mean_mse + std_mse,
                alpha=0.2, color='steelblue')
ax.axvline(lasso.alpha_, color='crimson', ls='--', lw=1.5,
           label=f'CV α = {lasso.alpha_:.4f}')
ax.axvline(best_alpha, color='orange', ls='--', lw=1.5,
           label=f'Strict α = {best_alpha:.4f}')
ax.set_xlabel('Alpha (log scale)')
ax.set_ylabel('Mean Squared Error')
ax.set_title('LassoCV — Alpha Selection Path')
ax.legend()
ax.invert_xaxis()
fig.tight_layout()
plt.show()